In [0]:
dbutils.widgets.text("catalog", "olist_dev")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = f"{CATALOG}.lakehouse"
print(f"schéma : {SCHEMA}")

In [0]:
%load_ext autoreload
%autoreload 2

import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from olist.bronze import ecrire_table
from olist.silver import TABLES_SILVER, CLES, problemes_de_cle, conversions_echouees

In [0]:
for nom, (fonction, entrees) in TABLES_SILVER.items():
    tables_bronze = [spark.table(f"{SCHEMA}.bronze_{e}") for e in entrees]
    ecrire_table(fonction(*tables_bronze), f"{SCHEMA}.silver_{nom}")
    print(f"écrite : {SCHEMA}.silver_{nom}")

In [0]:
erreurs = {}
for nom, (fonction, entrees) in TABLES_SILVER.items():
    silver = spark.table(f"{SCHEMA}.silver_{nom}")
    bronze = spark.table(f"{SCHEMA}.bronze_{entrees[0]}")
    problemes = problemes_de_cle(silver, CLES[nom])

    if nom != "geolocation":   # seule table qui change de grain
        if silver.count() != bronze.count():
            problemes.append(f"{silver.count():,} lignes au lieu de {bronze.count():,}")
        echecs = conversions_echouees(bronze, silver)
        if echecs:
            problemes.append(f"conversions échouées : {echecs}")

    print(f"silver_{nom:<14} {silver.count():>9,} lignes  "
          f"{'OK' if not problemes else 'ERREUR : ' + ' ; '.join(problemes)}")
    if problemes:
        erreurs[nom] = problemes

assert not erreurs, f"Silver incorrect : {erreurs}"

In [0]:
from pyspark.sql import functions as F 
orders = spark.table(f"{SCHEMA}.silver_orders")
mesure = orders.agg(
    F.sum(F.col("approbation_manquante").cast("int")).alias("approbation_manquante"),
    F.sum(F.col("transporteur_avant_approbation").cast("int")).alias("transporteur_avant_approbation"),
    F.sum(F.col("livree_avant_transporteur").cast("int")).alias("livree_avant_transporteur"),
    F.sum(F.col("statut_incoherent").cast("int")).alias("statut_incoherent"),
    F.count("en_retard").alias("commandes_avec_cible"),
    F.sum("en_retard").alias("retards"),
).first().asDict()

attendu = {
    "approbation_manquante": 160,
    "transporteur_avant_approbation": 1_359,
    "livree_avant_transporteur": 23,
    "statut_incoherent": 14,
    "commandes_avec_cible": 96_470,
    "retards": 6_534,
}
for cle, valeur in attendu.items():
    print(f"{cle:<32} attendu={valeur:>7,}  mesuré={mesure[cle]:>7,}  "
          f"{'OK' if mesure[cle] == valeur else 'ÉCART'}")

assert all(mesure[c] == v for c, v in attendu.items()), "Silver ne reproduit pas l'EDA"

In [0]:
display(spark.table(f"{SCHEMA}.silver_geolocation").orderBy(F.desc("nb_points")).limit(5))
display(spark.table(f"{SCHEMA}.silver_customers").limit(5))